[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/35_bpe_solution.ipynb)

# Solution: Byte-Pair Encoding (BPE)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
# No imports needed

In [ ]:
# ✅ SOLUTION

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def merge_pair(self, symbols, pair):
        result = []
        i = 0

        while i < len(symbols):
            if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
                result.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                result.append(symbols[i])
                i += 1

        return result

    def train(self, corpus, num_merges):
        vocab = {}

        for word in corpus:
            symbols = tuple(word) + ('</w>',)
            vocab[symbols] = vocab.get(symbols, 0) + 1

        self.merges = []

        for _ in range(num_merges):
            pairs = {}

            # Count pair frequencies
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    pairs[pair] = pairs.get(pair, 0) + freq

            if not pairs:
                break

            # Find most frequent pair
            best = max(pairs, key=pairs.get)
            self.merges.append(best)

            # Apply merge to vocabulary
            new_vocab = {}

            for word, freq in vocab.items():
                new_word = tuple(self.merge_pair(word, best))
                new_vocab[new_word] = new_vocab.get(new_word, 0) + freq

            vocab = new_vocab

    def encode(self, text):
        all_tokens = []

        for word in text.split():
            symbols = list(word) + ['</w>']

            # Apply learned merges
            for pair in self.merges:
                symbols = self.merge_pair(symbols, pair)

            all_tokens.extend(symbols)

        return all_tokens

In [ ]:
# Demo
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges)
print('Encode:', bpe.encode('low lower newest'))

In [ ]:
from torch_judge import check
check('bpe')